# 11. Final models for race and sex (Figure 5)

Same design as `05_final_model.ipynb` with race (`race_binary`: 1 = white, 2 = non-white) or sex (`sex`: 1 = male,
2 = female) as the target. The data are rebuilt from the raw GSS file with the preprocessing of
`08_race_sex_make_data.ipynb`, without the 5% missing-value filter and without list-wise deletion
(XGBoost native missing-value handling); rows are filtered only on the target.

Features: the 40 variables of each ballot in `inputs/top_40_features_m3_race.xlsx` / `inputs/top_40_features_m3_sex.xlsx`
(sheets `a_XGB`, `b_XGB`, `c_XGB`; see `10_race_sex_feature_selection.ipynb`), with `nom` variables as `category`.
As a safety net, the target-specific excluded variables of `09_race_sex_preliminary.ipynb` are removed even if
they appear in the xlsx.

Training: XGBoost, 5-fold CV × 20 repetitions, survey weights, SHAP.

**Input**: `data/gss7224_r3.dta`, `data/occ10-to-egp-class-crosswalk.csv`, `inputs/top_40_features_m3_{race,sex}.xlsx`

**Output**: `outputs/7_final_race_sex/7_FINAL_M3_Top40_{race_binary,sex}_*` (Figure 5)

## Data preparation

### Load data and merge survey-experiment items

In [ ]:
import pandas as pd
df_origin = pd.read_stata("data/gss7224_r3.dta", convert_categoricals=False)

# Merge survey-experiment versions of the same item
variable_groups = [
    # 1972-2018 vars
    # a
    ('amicable', 'amicably', 'amicablz'),
    # b
    ('bible', 'bibley'),
    ('blnumok', 'blnumoky'),
    # c
    ('class', 'classy'),
    ('clean', 'cleany', 'cleanz'),
    ('colaff', 'colaffy'),
    ('conarmy', 'conarmyy'),
    ('conbus', 'conbusy'),
    ('conclerg', 'conclergy'),
    ('coneduc', 'coneducy'),
    ('confed', 'confedy'),
    ('confinan', 'confinay'),
    ('conjudge', 'conjudgy'),
    ('conlabor', 'conlabory'),
    ('conlegis', 'conlegiy'),
    ('conmedic', 'conmediy'),
    ('conpress', 'conpresy'),
    ('consci', 'consciy'),
    ('consider', 'considey', 'considez'),
    ('control', 'controly', 'controlz'),
    ('contv', 'contvy'),
    ('courts', 'courtsy'),
    # d
    ('discaff', 'discaffy'),
    ('divlaw', 'divlawy'),
    # e
    ('eqwlth', 'eqwlthy'),
    # f
    ('fepol', 'fepoly'),
    # g
    ('genecanx', 'genecany'),
    ('getahead', 'getaheay'),
    ('grass', 'grassy'),
    # h
    ('hinumok', 'hinumoky'),
    ('honest', 'honesty', 'honestz'),
    # i
    ('interest', 'interesy', 'interesz'),
    # j
    ('judgment', 'judgmeny', 'judgmenz'),
    # m
    ('manners', 'mannersy', 'mannersz'),
    # n
    ('nataid', 'nataidy', 'nataidz'),
    ('natarms', 'natarmsy', 'natarmsz'),
    ('natcity', 'natcityy', 'natcityz'),
    ('natcrime', 'natcrimy', 'natcrimz'),
    ('natdrug', 'natdrugy', 'natdrugz'),
    ('nateduc', 'nateducy', 'nateducz'),
    ('natenvir', 'natenviy', 'natenviz'),
    ('natfare', 'natfarey', 'natfarez'),
    ('natheal', 'nathealy', 'nathealz'),
    ('natmass', 'natmassz'),
    ('natpark', 'natparkz'),
    ('natrace', 'natracey', 'natracez'),
    ('natroad', 'natroadz'),
    ('natsoc', 'natsocz'),
    ('natspac', 'natspacy', 'natspacz'),
    ('newsfrom', 'newsfrmy'),
    # o
    ('obeys', 'obeysy', 'obeysz'),
    # p
    ('polviews', 'polviewx', 'polviewy'),
    ('posslq', 'posslqy'),
    ('prayer', 'prayery', 'prayerx'),
    # r
    ('responsi', 'responsy', 'responsz'),
    ('role', 'roley', 'rolez'),
    # s
    ('scifrom', 'scifromy'),
    ('seeksci', 'seeksciy'),
    ('studious', 'studiouy', 'studiouz'),
    ('success', 'successy', 'successz'),
    # t
    ('trust', 'trusty'),
    # u
    ('uswar', 'uswary'),

    # 2024 vars
    # Traditional Wording Experiments
    ('marsame', 'marsamey'),
    ('posslq', 'posslqy'),
    ('racdif1', 'racdif1x'),
    ('racdif2', 'racdif2x'),
    ('racdif3', 'racdif3x'),
    ('racdif4', 'racdif4x'),
    ('racdif5', 'racdif5x'),
    ('xmovie', 'xmoviey'),

    # Volunteered Responses Experiments
    ('bible', 'biblev', 'biblenv'),
    ('courts', 'courtsv', 'courtsnv'),
    ('discaffw', 'discaffwv', 'discaffwnv'),
    ('divlaw', 'divlawv', 'divlawnv'),
    ('fair', 'fairv', 'fairnv'),
    ('fepol', 'fepolv', 'fepolnv'),
    ('fucitzn', 'fucitznv', 'fucitznnv'),
    ('getahead', 'getaheadv', 'getaheadnv'),
    ('grass', 'grassv', 'grassnv'),
    ('helpful', 'helpfulv', 'helpfulnv'),
    ('kidssol', 'kidssolv', 'kidssolnv'),
    ('postlife', 'postlifev', 'postlifenv'),
    ('prayer', 'prayerv', 'prayernv'),
    ('racopen', 'racopenv', 'racopennv'),
    ('reliten', 'relitenv', 'relitennv'),
    ('trust', 'trustv', 'trustnv'),
    ('uscitzn', 'uscitznv', 'uscitznnv'),

    # Gridded Variables Experiments
    ('abany', 'abanyg'),
    ('abdefect', 'abdefectg'),
    ('abhlth', 'abhlthg'),
    ('abnomore', 'abnomoreg'),
    ('abpoor', 'abpoorg'),
    ('abrape', 'abrapeg'),
    ('absingle', 'absingleg'),
    ('suicide1', 'suicide1g'),
    ('suicide2', 'suicide2g'),
    ('suicide4', 'suicide4g'),

    # Gender-neutral Wording Experiments
    ('colcom', 'colcomy'),
    ('famdif16', 'famdif16y'),
    ('letdie1', 'letdie1y'),
    ('libath', 'libathy'),
    ('libcom', 'libcomy'),
    ('libmslm', 'libmslmy'),
    ('librac', 'libracy'),
    ('polabuse', 'polabusey'),
    ('polattak', 'polattaky'),
    ('polhitok', 'polhitoky'),
    ('spkath', 'spkathy'),
    ('spkcom', 'spkcomy'),
    ('spkmslm', 'spkmslmy'),
    ('spkrac', 'spkracy'),

    # found later
    ('libhomo', 'libhomoy'),
    ('spkhomo', 'spkhomoy')
]

# --- Merge each group into one variable ---
df1 = df_origin.copy()
original_cols_to_drop = set() # set avoids duplicates

for group in variable_groups:
    # Keep only the group members present in the data
    existing_vars = [var for var in group if var in df1.columns]

    if not existing_vars:
        continue

    # New column name from the first variable of the group (e.g. nataid_c)
    base_name = existing_vars[0]
    combined_col = f"{base_name.lower()}_c"

    # Combine
    # Start from the first variable and fill its missing values with the other versions
    combined_series = df1[existing_vars[0]]
    for i in range(1, len(existing_vars)):
        combined_series = combined_series.combine_first(df1[existing_vars[i]])

    # Add the merged series to the data
    df1[combined_col] = combined_series

    # Mark the original columns for removal
    original_cols_to_drop.update(existing_vars)

# --- Drop the original (unmerged) variables ---
df1 = df1.drop(columns=list(original_cols_to_drop))

### Target variable, EGP class coding, split by year

In [ ]:
import pandas as pd
import numpy as np

df = df1.copy()
# No partyid filter: for the race/sex data only respondents with a missing target are dropped (below)

# race_binary: race 1 (white) -> 1; race 2 (black) / 3 (other) -> 2; missing stays missing
df['race_binary'] = df['race'].map({1: 1, 2: 2, 3: 2})  # 1 = white, 2 = non-white (0 not used, compatible with the `!= 0` filter)

# Years of schooling are dropped; education is represented by degree only
eduyear = ['educ', 'speduc', 'paeduc', 'maeduc']

# Occupation codes are harmonized to the 2010 Census scheme

# Convert occ10 to EGP class
# --- Convert occ10 and parallel variables to EGP class (based on GSS MR125) ---
print("\n--- EGP class conversion: start ---")
crosswalk_file = 'data/occ10-to-egp-class-crosswalk.csv'
try:
    df_egp_crosswalk = pd.read_csv(crosswalk_file)
    print(f"Loaded '{crosswalk_file}'.")

    # Make crosswalk occ10 numeric
    df_egp_crosswalk['occ10'] = pd.to_numeric(df_egp_crosswalk['occ10'], errors='coerce')
    # Keep only the EGP 10-class code for the merge
    crosswalk_mini = df_egp_crosswalk[['occ10', 'egp10_10']].copy()

    # --- 1. Respondent (R): 10-, 11- and 12-class versions ---
    print("Processing Respondent (R)...")
    df['occ10'] = pd.to_numeric(df['occ10'], errors='coerce')
    df = pd.merge(df, crosswalk_mini, on='occ10', how='left')

    # Classes reassigned to class IV (self-employed): IIIa, IIIb, V, VI, VIIa
    egp_base_classes_for_iv = [3, 4, 8, 9, 10]

    # Respondent's EGP 11-class and 12-class
    if all(col in df.columns for col in ['wrkslf', 'numemps', 'egp10_10']):
        df['wrkslf'] = pd.to_numeric(df['wrkslf'], errors='coerce')
        df['numemps'] = pd.to_numeric(df['numemps'], errors='coerce')

        # 11-Class (egp10_11)
        df['egp10_11'] = df['egp10_10']
        condition_11 = (df['egp10_10'].isin(egp_base_classes_for_iv) & (df['wrkslf'] == 1))
        df.loc[condition_11, 'egp10_11'] = 5 # 5 = class IVab

        # 12-Class (egp10_12)
        df['egp10_12'] = df['egp10_10']
        condition_12a = (condition_11 & (df['numemps'] > 0)) # employers
        condition_12b = (condition_11 & (df['numemps'] == 0)) # own-account workers
        df.loc[condition_12a, 'egp10_12'] = 5 # 5 = class IVa
        df.loc[condition_12b, 'egp10_12'] = 6 # 6 = class IVb
        print("Respondent (R): EGP 10-, 11- and 12-class created.")
    else:
        print("Warning: respondent 'wrkslf' or 'numemps' not found; skipping 11/12-class.")

    # --- 2. Parallel variables (spouse, parents, etc.): up to 11-class ---
    # (occ_code, self_employ_code, new_egp10_col_name, new_egp11_col_name)
    individuals_to_process = {
        'sp': ('spocc10', 'spwrkslf', 'sp_egp10_10', 'sp_egp10_11'),
        'pa': ('paocc10', 'pawrkslf', 'pa_egp10_10', 'pa_egp10_11'),
        'ma': ('maocc10', 'mawrkslf', 'ma_egp10_10', 'ma_egp10_11'),
        'kd': ('kdocc10', 'kdwrkslf', 'kd_egp10_10', 'kd_egp10_11'),
        'sb': ('sbocc10', 'sbwrkslf', 'sb_egp10_10', 'sb_egp10_11'),
        'sphed': ('sphedocc10', 'sphedslf', 'sphed_egp10_10', 'sphed_egp10_11'), # spouse's parent uses 'sphedslf'
    }

    for prefix, (occ_col, slf_col, egp10_col, egp11_col) in individuals_to_process.items():
        if occ_col in df.columns:
            print(f"--- Processing {prefix} ({occ_col}) ---")

            # Harmonize type
            df[occ_col] = pd.to_numeric(df[occ_col], errors='coerce')

            # Step A: merge the 10-class code
            temp_crosswalk = crosswalk_mini.rename(columns={'occ10': occ_col, 'egp10_10': egp10_col})
            df = pd.merge(df, temp_crosswalk, on=occ_col, how='left')

            # Step B: 11-class code (when the self-employment variable 'slf_col' exists)
            if slf_col in df.columns:
                df[slf_col] = pd.to_numeric(df[slf_col], errors='coerce')
                df[egp11_col] = df[egp10_col] # copy the 10-class code

                condition_11_parallel = (
                    df[egp10_col].isin(egp_base_classes_for_iv) &
                    (df[slf_col] == 1)
                )
                df.loc[condition_11_parallel, egp11_col] = 5 # 5 = class IVab
                print(f"Created '{egp10_col}' and '{egp11_col}'.")
            else:
                print(f"Warning: '{slf_col}' not found; creating '{egp10_col}' only.")
        else:
            print(f"Skipping {prefix}: '{occ_col}' not found.")

    print("--- EGP class conversion: done ---")

except FileNotFoundError:
    print(f"Error: {crosswalk_file} not found.")
except KeyError as e:
    print(f"Error: required column not found. {e}")
except Exception as e:
    print(f"Unexpected error: {e}")
# --- End of EGP conversion ---


# Keep prestg10 and sei10; drop the other occupation codes
occupation = [
    # === 1. Respondent (R) legacy variables (1970, 1980, ISCO, DOT) ===
    'occ', 'industry', 'prestige', 'sei', 'occ80', 'indus80', 'prestg80', 'isco68', 'isco681', 'isco88',
    'dotdata', 'dotpeop', 'dotthng', 'dotged', 'dotsvp', 'dotpres', 'occfirst', 'presfrst', 'wrkslfst', 'indfirst',

    # === 2. Spouse legacy variables ===
    'spocc', 'spind', 'sppres', 'spsei', 'spocc80', 'spind80', 'sppres80',
    'spisco68', 'spisc681', 'spisco88', 'spdotdat', 'spdotpeo', 'spdotthn', 'spdotged', 'spdotsvp', 'spdotpre',

    # === 3. Parents legacy variables ===
    'paocc16', 'paind16', 'papres16', 'paocc80', 'paind80', 'papres80', 'pasei',
    'paisc681', 'paisco68', 'paisco88', 'padotdat', 'padotpeo', 'padotthn', 'padotged', 'padotsvp', 'padotrpe',
    'maocc80', 'maind80', 'mapres80', 'masei', 'maisc681', 'maisco88',

    # === 4. Other family legacy variables (children, siblings, spouse's parents) ===
    'kdocc80', 'kdind80', 'kdpres80', 'sbocc80', 'sbind80', 'sbpres80', 'sphedocc', 'sphedind', 'sphedpre',

    # === 5. Alternative prestige scales (...105PLUS) ===
    'prestg105plus', 'sppres105plus', 'papres105plus', 'mapres105plus', 'kdpres105plus', 'sbpres105plus', 'sphedpres105plus',

    # === 6. Metadata / flag variables (not used for modelling) ===
    # Flags for the existence of verbatim text
    'occindv', 'spoccindv', 'paoccindv', 'maoccindv', 'kdoccindv', 'sboccindv', 'sphedoccindv',

    # Coding status flags
    'occstatus', 'indstatus', 'spoccstatus', 'spindstatus', 'paoccstatus', 'paindstatus', 'maoccstatus',
    'maindstatus', 'kdoccstatus', 'kdindstatus', 'sboccstatus', 'sbindstatus', 'sphedoccstatus', 'sphedindstatus',

    # Coding tag flags
    'occtag', 'indtag', 'spocctag', 'spindtag', 'paocctag', 'paindtag', 'maocctag', 'maindtag',
    'kdoctag', 'kdindtag', 'sboctag', 'sbindtag', 'sphedocctag', 'sphedindtag',

    # === 7. Original ...10 codes replaced by EGP ===
    # (industry codes such as indus10 and indus07 included)
    'occ10', 'indus10', 'indus07', 'isco08',
    'spocc10', 'spind10',
    'paocc10', 'paind10',
    'maocc10', 'maind10',
    'kdocc10', 'kdind10',
    'sbocc10', 'sbind10',
    'sphedocc10', 'sphedind10',

    # EGP variables not used
    'egp10_10', 'egp10_12',
    'spocc10', 'spwrkslf', 'sp_egp10_10',
    'paocc10', 'pawrkslf', 'pa_egp10_10',
    'maocc10', 'mawrkslf', 'ma_egp10_10',
    'kdocc10', 'kdwrkslf', 'kd_egp10_10',
    'sbocc10', 'sbwrkslf', 'sb_egp10_10',
    'sphedocc10', 'sphedslf', 'sphed_egp10_10' # spouse's parent uses 'sphedslf'
]

# Detailed Protestant denomination variable
protestant = ['others']

features_to_drop = [col for col in occupation if col in df.columns] +\
                [col for col in eduyear if col in df.columns] +\
                [col for col in protestant if col in df.columns]

df = df.drop(columns=features_to_drop, errors='ignore')

# --- Dictionary by year (all rows kept; only entirely missing columns dropped) ---
# Unlike 08_race_sex_make_data.ipynb, no target pre-filter, no 5% column filter, and no row-wise deletion.
# (Rows are filtered on the target in the model function; missing feature values are handled natively by XGBoost.)
df_by_year = {year: data.dropna(axis=1, how='all')
              for year, data in df.groupby('year')}
print(f"df_by_year created: {len(df_by_year)} years")

### Restrict to the selected variables (with leakage safety net) and set dtypes

In [ ]:
# --- Load features and types from the ballot x model sheets of the xlsx -> data_for_analysis ---
# ===== Feature sources =====
RACE_XLSX = 'inputs/top_40_features_m3_race.xlsx'
SEX_XLSX  = 'inputs/top_40_features_m3_sex.xlsx'
ESSENTIAL_COLS = ['wtssnrps', 'wtssps', 'ballot']

import os
import pandas as pd
import numpy as np

RACE_POTENTIAL_TARGETS = [
    'race', 'race_binary', 'racecen1', 'racecen2', 'racecen3', 'hispanic', 'hisporig', 'hhrace',
    'racecen1_0022', 'racecen2_0022', 'racecen3_0022', 'hispanic_0022', 'racerank1', 'racerank2', 'racerank3', 'raceacs1',
    'raceacs2', 'raceacs3', 'raceacs4', 'raceacs5', 'raceacs6', 'raceacs7', 'raceacs8', 'raceacs9',
    'raceacs10', 'raceacs11', 'raceacs12', 'raceacs13', 'raceacs14', 'raceacs15', 'raceacs16', 'raceacs14_2122',
    'ethnic', 'eth1', 'ethworld1', 'ethworld2', 'ethworld3', 'ethworld4', 'ethworld5', 'ethworld6',
    'ethworld7', 'ethworld8', 'ethworld9', 'ethregion1', 'ethregion2', 'ethregion3', 'ethregion4', 'ethregion5',
    'ethregion6', 'ethregion7', 'ethregion8', 'ethregion9', 'ethregion10', 'ethregion11', 'ethregion12', 'ethregion13',
    'ethregion14', 'ethregion15', 'ethregion16', 'ethregion17', 'ethregion18', 'ethregion19', 'ethregion20', 'ethregion21',
    'ethregion22', 'ethregion23', 'ethregion24', 'ethregion25', 'ethregion26', 'ethregion27', 'ethregion28', 'ethregion29',
    'ethregion30', 'ethregion31', 'ethregion32', 'ethregion33', 'ethregion34', 'ethregion35', 'ethregion36', 'ethregion37',
    'ethregion38', 'ethregion39', 'ethregion40', 'ethregion41', 'ethregion42', 'ethregion43', 'ethregion44', 'ethregion45',
    'ethregion46', 'ethregion47', 'ethregion48', 'ethregion49', 'ethregion50', 'ethregion51', 'ethregion52', 'ethregion53',
    'ethregion54', 'ethregion55', 'ethregion56', 'ethregion57', 'ethregion58', 'ethregion59', 'ethregion60', 'ethregion61',
    'ethregion62', 'ethregion63', 'ethregion64', 'ethregion65', 'ethregion66', 'ethregion67', 'ethregion68', 'ethregion69',
    'ethregion70', 'ethregion71', 'ethregion72', 'ethregion73', 'ethregion74', 'ethregion75', 'ethregion76', 'ethregion77',
    'ethregion78', 'ethregion79', 'ethregion80', 'ethregion81', 'ethregion82', 'ethregion83', 'ethregion84', 'ethregion85',
    'ethregion86', 'ethregion87', 'ethregion88', 'ethregion89', 'ethregion90', 'ethregion91', 'ethregion92', 'ethregion93',
    'ethregion94', 'ethregion95', 'racesee', 'racedbtf', 'racethwh', 'racethbl', 'racethhi', 'racethas',
    'racethna', 'racethot', 'ethimp', 'ethnum', 'chldfam_wht', 'chldfam_blk', 'chldfam_asn', 'chldfam_hisp',
    'chldfam_oth', 'closeblk', 'marblk', 'liveblks',
]
SEX_POTENTIAL_TARGETS = [
    'sex', 'sexbirth1', 'sexnow1', 'sexnow2', 'gender1', 'rplace', 'rplace_7522', 'femsee1',
    'femself1', 'mascsee1', 'mascself1', 'rheight',
]
NOT_NEEDED = [
    'hlthstrt', 'huadd', 'huaddwhy', 'dwellpre', 'kidsinhh', 'respond', 'incuspop', 'neisafe',
    'rlooks', 'rgroomed', 'rweight', 'rhlthend', 'huclean', 'coop', 'comprend', 'consent',
    'adminconsent', 'whoelse1', 'whoelse2', 'whoelse3', 'whoelse4', 'whoelse5', 'whoelse6', 'lngthinv',
    'intage', 'intage_0418', 'intethn', 'intsex', 'intyrs', 'intyrs_0418', 'inthisp', 'intrace1',
    'intrace2', 'intrace3', 'ratetone', 'mode', 'modesequence', 'feeused', 'feelevel', 'totalincentive',
    'spaneng', 'spanint', 'spanself', 'formwt', 'oversamp', 'wtss', 'wtssall', 'wtssnr',
    'wtsps', 'wtssps', 'wtssnrps', 'ballotformwt', 'ballotformwtnr', 'wtsps_as', 'wtssnrps_as', 'id',
    'hefinfo', 'hefinfo1', 'respnum', 'respnumh', 'year', 'dateintv', 'ballot', 'version',
    'form', 'issp', 'phone', 'sample', 'sampcode', 'phase', 'subsamprate', 'batch',
    'kish', 'vpsu', 'vstrat', 'fileversion', 'final_weight', 'intid', 'genevig1', 'genevig2',
    'mostimp1', 'mostimp2', 'mostimp3', 'mostimp4', 'prob1', 'prob2', 'prob3', 'prob4',
    'extra1', 'extra2', 'srcbelt', 'size', 'babies_7222', 'earnrs_7222', 'hompop_7222', 'preteen_7222',
    'reg16_7222', 'region_7222', 'sibs_7222', 'teens_7222', 'saqsex', 'saqissp', 'easyget', 'scimode',
    'sciintro', 'rundrstd', 'robject', 'intrwght', 'saqgene', 'coop2', 'hhtype',
]

ATTITUDE_RACE = [
    'racdif1', 'racdif2', 'racdif3', 'racdif4', 'racdif5', 'racdif1_c', 'racdif2_c', 'racdif3_c',
    'racdif4_c', 'racdif5_c', 'racdif1x', 'racdif2x', 'racdif3x', 'racdif4x', 'racdif5x', 'natrace',
    'natrace_c', 'natracey', 'natracez', 'discaff', 'discaff_c', 'discaffy', 'affrmact', 'affrmact2',
    'helpblk', 'wrkwayup', 'busing', 'feelblks', 'feelwhts', 'closewht', 'racopen', 'racopen_c',
    'racmar', 'racdin', 'racpush', 'racseg', 'racfew', 'rachaf', 'racmost', 'raclive',
    'racpres', 'racschol', 'racwork', 'racchurh', 'racinteg', 'racchng', 'dsndntpay', 'workblks',
    'workwhts', 'intlblks', 'intlwhts', 'wlthblks', 'wlthwhts', 'fareblks', 'farewhts', 'rachome',
    'marwht', 'livewhts', 'blkwhite', 'blkjobs', 'blkhouse', 'blksimp', 'whtsdiff',
]
ATTITUDE_SEX = [
    'fepol', 'fepol_c', 'fepoly', 'fework', 'fechld', 'fechld2', 'fepresch', 'fepresch2',
    'fefam', 'fehome', 'fehelp', 'fehire', 'fejobaff', 'meovrwrk', 'discaffw', 'discaffw_c',
    'discaffm', 'homosex', 'homosex1', 'marhomo', 'premarsx', 'teensex', 'xmarsex', 'xmarsex1',
    'pillok', 'sexeduc', 'feovrwrk', 'hubbywrk', 'hubbywk', 'hubbywk1', 'twoincs', 'twoincs1',
    'gendereq', 'fepres',
]

sheet_key_mapping = {
    'a_XGB': ('a', 'XGBoost'),
    'b_XGB': ('b', 'XGBoost'),
    'c_XGB': ('c', 'XGBoost'),
}

def load_top_features(excel_file_name):
    """con/nom feature lists of each (ballot, model) sheet according to the 'type' column."""
    if not os.path.exists(excel_file_name):
        raise FileNotFoundError(f"Excel file '{excel_file_name}' not found.")
    top = {}
    for sheet_name, dict_key in sheet_key_mapping.items():
        dfx = pd.read_excel(excel_file_name, sheet_name=sheet_name)
        if 'feature' not in dfx.columns or 'type' not in dfx.columns:
            raise ValueError(f"{excel_file_name} sheet '{sheet_name}' lacks 'feature'/'type' columns.")
        con_list = dfx.loc[dfx['type'] == 'con', 'feature'].dropna().tolist()
        nom_list = dfx.loc[dfx['type'] == 'nom', 'feature'].dropna().tolist()
        other = dfx.loc[~dfx['type'].isin(['con', 'nom']), 'feature'].dropna().tolist()
        if other:
            print(f"  Warning {dict_key}: types other than con/nom {other} -> treated as nom")
            nom_list += other
        top[dict_key] = {'con': con_list, 'nom': nom_list}
        print(f"  {excel_file_name} | {sheet_name} {dict_key}: con {len(con_list)}, nom {len(nom_list)}")
    return top


def build_data_dict(df_by_year, top_features, target_col, essential_cols):
    """Restrict df_by_year to the top features of each (ballot, model) and encode nom as category.
    Returns {(ballot, model): {year: df}} (same structure as 05_final_model.ipynb).

    [Leakage safety net, as in build_selective_data_dict of 10_race_sex_feature_selection.ipynb] The target-specific
    excluded variables (potential_targets + attitudes + not_needed) are removed from the features,
    so a leakage variable included in the xlsx by mistake never enters the model."""
    if target_col == 'race_binary':
        exclusion_base = set(RACE_POTENTIAL_TARGETS + ATTITUDE_RACE + NOT_NEEDED)
    elif target_col == 'sex':
        exclusion_base = set(SEX_POTENTIAL_TARGETS + ATTITUDE_SEX + NOT_NEEDED)
    else:
        raise ValueError(f"Unsupported target_col: '{target_col}' (race_binary or sex)")

    dropped_by_guard = set()
    data = {}
    for year, df_year_original in df_by_year.items():
        if year < 1988 or 'ballot' not in df_year_original.columns:
            continue
        df_year = df_year_original.copy()
        if df_year.columns.has_duplicates:
            df_year = df_year.loc[:, ~df_year.columns.duplicated()]
        for ballot_code, ballot_label in zip([1, 2, 3], ['a', 'b', 'c']):
            df_b = df_year[df_year['ballot'] == ballot_code].copy()
            if df_b.empty:
                continue
            for model_name in ['XGBoost']:
                key = (ballot_label, model_name)
                if key not in top_features:
                    continue
                con_raw = top_features[key]['con']
                nom_raw = top_features[key]['nom']
                con_f = [f for f in con_raw if f not in exclusion_base]
                nom_f = [f for f in nom_raw if f not in exclusion_base]
                dropped_by_guard.update(f for f in (con_raw + nom_raw) if f in exclusion_base)
                feat_all = con_f + nom_f
                keep = list(dict.fromkeys(feat_all + [target_col] + essential_cols))  # order-preserving: column order must not depend on the per-process string hash
                keep = [c for c in keep if c in df_b.columns]
                if target_col not in keep:
                    raise ValueError(f"{key}-{year}: target '{target_col}' not in the data.")
                dff = df_b.loc[:, keep].copy()
                for col in dff.columns:
                    if col in nom_f:
                        dff[col] = dff[col].astype('category')
                    elif col in con_f or col == target_col:
                        dff[col] = pd.to_numeric(dff[col], errors='coerce')
                if not dff.empty:
                    data.setdefault(key, {})[year] = dff
    if dropped_by_guard:
        print(f"  [safety net] {target_col}: removed {len(dropped_by_guard)} leakage variables from the xlsx -> {sorted(dropped_by_guard)}")
    else:
        print(f"  [safety net] {target_col}: no leakage variables removed")
    return data


print('--- Loading race features ---')
top_race = load_top_features(RACE_XLSX)
print('--- Loading sex features ---')
top_sex = load_top_features(SEX_XLSX)

data_for_analysis_race = build_data_dict(df_by_year, top_race, 'race_binary', ESSENTIAL_COLS)
data_for_analysis_sex  = build_data_dict(df_by_year, top_sex,  'sex',         ESSENTIAL_COLS)

print(f"\nrace: {len(data_for_analysis_race)} (ballot, model) groups")
print(f"sex : {len(data_for_analysis_sex)} (ballot, model) groups")

# Example check
ck = ('a', 'XGBoost')
if ck in data_for_analysis_race and data_for_analysis_race[ck]:
    yy = sorted(data_for_analysis_race[ck])[0]
    print(f"\nExample race {ck} {yy}: shape {data_for_analysis_race[ck][yy].shape}")

### Save / load the model data (optional)

In [ ]:
# --- (optional) Save data_for_analysis to skip rebuilding from the .dta file ---
import pickle
with open('data/7_data_for_final_race.pkl', 'wb') as f:
    pickle.dump(data_for_analysis_race, f)
with open('data/7_data_for_final_sex.pkl', 'wb') as f:
    pickle.dump(data_for_analysis_sex, f)
print('Saved: 7_data_for_final_race.pkl / 7_data_for_final_sex.pkl')

In [ ]:
# --- (optional) Load the saved data_for_analysis instead of rebuilding from the .dta file ---
import pickle, os
if os.path.exists('data/7_data_for_final_race.pkl') and os.path.exists('data/7_data_for_final_sex.pkl'):
    with open('data/7_data_for_final_race.pkl', 'rb') as f:
        data_for_analysis_race = pickle.load(f)
    with open('data/7_data_for_final_sex.pkl', 'rb') as f:
        data_for_analysis_sex = pickle.load(f)
    print('Loaded: race/sex data_for_analysis')
else:
    print('Saved files not found. Run the data preparation cells above.')

## Prediction model

In [ ]:
import os

# Output folder of this notebook
OUT_DIR = "outputs/7_final_race_sex"

import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import os

# --- Core function: train/evaluate a model for every (ballot, model, year) ---
def run_single_ml_iteration_weighted_top(
    data_for_analysis_dict, # input data: {(ballot, model): {year: DataFrame}}
    target_col='partyid_1', # target column
    random_seed=123,        # random seed for the model and KFold (set differently in each run)
    run_number=1,           # run index; out-of-fold SHAP values are saved only when it is 1
    use_weights=True,       # apply GSS weights (wtssnrps/wtssps) to training and evaluation
    output_folder_name=None # folder for the out-of-fold SHAP files (None = current folder)
):
    """
    For each (ballot, model) and year, train and evaluate the model with 5-fold CV
    and return the SHAP importance averaged over folds.

    When run_number == 1, the test-fold SHAP values of all folds are also collected,
    reordered to the original row order, and saved as out-of-fold (OOF) SHAP values in output_folder_name.
    (This gives SHAP values for the full sample; run 1 is sufficient.)
    """
    scores_list = []
    importances_list = []
    dimensions_list = []
    feature_usage_list = []

    # Keys are (ballot, model); values are {year: DataFrame}.
    for (ballot, model_name), year_dict in data_for_analysis_dict.items():

        # The dictionary must contain only the 'XGBoost' model (guaranteed by the data preparation).
        # Any other value means the upstream logic is broken, so stop immediately.
        if model_name not in ['XGBoost']:
            raise ValueError(f"Unknown model_name: '{model_name}' (ballot={ballot})")

        # Samples and features differ by year, so a separate model is trained for every year.
        for year, df_original in year_dict.items():
            dataset_identifier = f"({ballot}, {model_name}) - {year}"  # identifier for logs/errors

            # OOF SHAP file names (used only in run 1)
            safe_filename_base = f"SHAP_OOF_run1_{target_col}_{ballot}_{model_name}_{year}"

            # Copy so the input is not modified. Duplicated index values could make the .loc-based
            # indexing below select the wrong rows, so reset to 0..n-1.
            df = df_original.copy()
            if not df.index.is_unique: df.reset_index(drop=True, inplace=True)

            # The data preparation guarantees that this data frame is non-empty and contains
            # target_col. If not, the upstream data are broken, so stop.
            #
            if df.empty:
                raise ValueError(f"{dataset_identifier} - empty data frame.")
            if target_col not in df.columns:
                raise ValueError(f"{dataset_identifier} - target column '{target_col}' not found.")

            # partyid_1: 1 = Democrat, 2 = Republican, 0 = independent/other (not part of this binary task).
            # Respondents with a missing or 0 target are dropped.
            df_filtered = df[df[target_col].notna() & (df[target_col] != 0)].copy()
            # No cases after filtering would mean no Democrats/Republicans in this year/ballot;
            # stop rather than silently dropping the year from the results.
            if df_filtered.empty:
                raise ValueError(f"{dataset_identifier} - no cases left after filtering '{target_col}' != 0.")

            # Re-index to 0..n-1 after filtering
            # (used below to restore the original order of the OOF results from the KFold positions)
            df_filtered.reset_index(drop=True, inplace=True)

            # Target: 1 = Democrat, 2 = Republican
            y_original = df_filtered[target_col]

            # GSS weights make the sample represent the population in training and evaluation.
            # wtssnrps from 2004 on, wtssps before; equal weights (1) if neither exists.
            weights_original = pd.Series(np.ones(len(df_filtered)), index=df_filtered.index)
            if use_weights:
                if 'wtssnrps' in df_filtered.columns and pd.to_numeric(year, errors='coerce') >= 2004:
                    weights_original = df_filtered['wtssnrps']
                elif 'wtssps' in df_filtered.columns:
                    weights_original = df_filtered['wtssps']
                weights_original = weights_original.fillna(1)

            # 5-fold CV needs at least 20 cases (>= 4 per fold)
            # and both classes (Democrat and Republican). Otherwise raise an error
            # instead of silently skipping, so the sample design (year/subgroup) can be checked.
            if len(df_filtered) < 20 or y_original.nunique() < 2:
                raise ValueError(
                    f"{dataset_identifier} - too few cases ({len(df_filtered)}) or "
                    f"too few classes of '{target_col}' ({y_original.nunique()})."
                )

            # Columns excluded from the features (X):
            # - potential_targets: the target itself (avoid leakage)
            # - nonessential_cols: wtssnrps/wtssps (weights, already extracted above),
            #                      ballot/zodiac (survey-design metadata / meaningless variable)
            potential_targets = [target_col]
            nonessential_cols = ['wtssnrps', 'wtssps', 'ballot', 'zodiac']

            all_cols_to_drop = potential_targets + nonessential_cols
            # Columns differ by year, so only columns present in df_filtered are dropped.
            features_to_drop = [col for col in set(all_cols_to_drop) \
                                if col in df_filtered.columns]

            X = df_filtered.drop(columns=features_to_drop, errors='ignore')


            # 'nom' (categorical) features were already converted to category dtype in data preparation,
            # so the dtype identifies categorical vs. continuous features.
            categorical_cols = X.select_dtypes(include='category').columns.tolist()
            # X only drops columns from df_filtered and keeps its index,
            # so y_original and weights_original can be used as they are.
            y_encoded = LabelEncoder().fit_transform(y_original)  # {1,2} -> {0,1}
            n_classes = len(np.unique(y_encoded))  # always 2 (Democrat/Republican)
            feature_names = X.columns.tolist()
            # --- End of data preparation ---


            # --- Model ---
            # enable_categorical=True: columns with category dtype are handled natively as categorical
            if model_name == 'XGBoost':
                model = xgb.XGBClassifier(
                    eval_metric='mlogloss',
                    random_state=random_seed,
                    enable_categorical=True
                )
            else:
                # model_name was validated at the top, so this branch cannot be reached.
                # If it is, the upstream logic is broken, so raise an error.
                raise ValueError(f"Unknown model_name: '{model_name}'")
            # --- End of model ---

            # In run 1, the test-fold SHAP values/data/indices of each fold are collected
            # and saved as out-of-fold (OOF) results after the loop.
            if run_number == 1:
                oof_shap_list = []
                oof_xtest_list = []
                oof_indices_list = []

            # 5-fold CV: collect scores (accuracy/f1/roc_auc) and SHAP importance in every fold.
            kfold = KFold(n_splits=5, shuffle=True, random_state=random_seed)
            fold_scores = {'accuracy': [], 'f1_macro': [], 'roc_auc_ovr': []}
            fold_importances = []

            for fold_i, (train_index_rel, test_index_rel) in enumerate(kfold.split(X, y_encoded)):
                # kfold.split returns positions 0..n-1; they are mapped through X.index
                # to index values (identical to positions here) and rows are selected with .loc.
                train_indices_abs = X.index[train_index_rel]
                test_indices_abs = X.index[test_index_rel]

                X_train, X_test = X.loc[train_indices_abs], X.loc[test_indices_abs]
                y_train, y_test = y_encoded[train_index_rel], y_encoded[test_index_rel]
                weights_train, weights_test = weights_original.loc[train_indices_abs], weights_original.loc[test_indices_abs]

                # --- Train ---
                fit_params = {'sample_weight': weights_train} if use_weights else {}
                # A training failure (e.g. y_train with a single class, so base_score is undefined)
                # means the fold/year/subgroup sample is unsuitable for the analysis.
                # Catching it and skipping silently would average over fewer folds (e.g. 4 instead of 5)
                # without any trace in the result files,
                # so the exception is propagated to stop the analysis.
                model.fit(X_train, y_train, **fit_params)


                # --- Predict and score ---
                # Silently ignoring errors in prediction/scoring (e.g. roc_auc_score undefined for a
                # single-class y_test) would average only some metrics of fold_scores,
                # which is hard to notice in the result files.
                # The exception is therefore propagated.
                y_pred = model.predict(X_test)
                y_proba = model.predict_proba(X_test)

                fold_scores['accuracy'].append(accuracy_score(y_test, y_pred, sample_weight=weights_test))
                fold_scores['f1_macro'].append(f1_score(y_test, y_pred, average='macro', sample_weight=weights_test))

                if n_classes > 2:
                    fold_scores['roc_auc_ovr'].append(roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro', sample_weight=weights_test))
                else:
                    fold_scores['roc_auc_ovr'].append(roc_auc_score(y_test, y_proba[:, 1], average='macro', sample_weight=weights_test))
                # --- End of prediction/scoring ---


                # --- SHAP ---
                # SHAP values measure each feature's contribution to the model's predictions;
                # the mean absolute value per fold is used as feature importance.
                # With >= 20 cases and 5 folds, X_test is never empty.
                # If it is, the upstream logic is broken, so raise instead of filling with NaN.
                if X_test.empty:
                    raise ValueError(f"{dataset_identifier} Fold {fold_i} - X_test is empty.")

                # A SHAP failure indicates a structural problem with the model/input data,
                # so the exception is propagated.
                explainer = shap.TreeExplainer(
                    model,
                    feature_perturbation='tree_path_dependent'
                )

                shap_values_output = explainer.shap_values(
                    X_test,
                    check_additivity=False
                )
                # --- End of SHAP ---


                # --- Harmonize the output format of shap_values() ---
                shap_values_to_process = None

                if isinstance(shap_values_output, list) and len(shap_values_output) == n_classes:
                    # (class 0 = Democrat, class 1 = Republican)
                    # SHAP values for predicting Republican (1)
                    shap_values_to_process = shap_values_output[1]
                elif isinstance(shap_values_output, np.ndarray):
                    # (binary model returning a single array)
                    shap_values_to_process = shap_values_output

                # Unexpected SHAP formats or dimensions indicate a problem with the upstream data/model,
                # so raise instead of filling with NaN.
                if shap_values_to_process is None:
                    raise ValueError(
                        f"{dataset_identifier} Fold {fold_i} - SHAP values missing or "
                        f"in an unexpected format. (Type: {type(shap_values_output)})"
                    )

                mean_abs_shap = np.abs(shap_values_to_process).mean(axis=0)
                if len(mean_abs_shap) != len(feature_names):
                    raise ValueError(
                        f"{dataset_identifier} Fold {fold_i} - SHAP dimension mismatch. "
                        f"(SHAP: {len(mean_abs_shap)}, Feat: {len(feature_names)})"
                    )
                fold_importances.append(mean_abs_shap)

                # Keep OOF data (run 1 only)
                if run_number == 1:
                    oof_shap_list.append(shap_values_to_process)
                    oof_xtest_list.append(X_test)
                    oof_indices_list.append(test_indices_abs)
                # --- End of SHAP processing ---

            # --- End of KFold loop ---

            # --- Combine and save OOF data (after the KFold loop, run 1 only) ---
            # Failures when combining/saving OOF SHAP values (array or file IO errors) affect
            # the integrity of the outputs, so the exception is propagated.
            # If the fold loop finished without errors, oof_shap_list is always filled.
            if run_number == 1:
                # 1. Concatenate the fold pieces
                combined_shap = np.concatenate(oof_shap_list, axis=0)
                combined_xtest = pd.concat(oof_xtest_list, axis=0)
                combined_indices = np.concatenate(oof_indices_list, axis=0)

                # 2. Restore the original row order
                sort_order = np.argsort(combined_indices)
                final_oof_shap_values = combined_shap[sort_order]
                final_oof_xtest = combined_xtest.iloc[sort_order]

                # 3. Create the folder and file paths
                output_dir = "." # default: current folder
                if output_folder_name:
                    output_dir = output_folder_name
                    if not os.path.exists(output_dir):
                        os.makedirs(output_dir)
                        print(f"--- Creating OOF folder: {output_dir} ---")

                # Build paths with os.path.join
                shap_filename = os.path.join(output_dir, f"{safe_filename_base}_values.npy")
                xtest_filename = os.path.join(output_dir, f"{safe_filename_base}_xtest.parquet")

                np.save(shap_filename, final_oof_shap_values)
                final_oof_xtest.to_parquet(xtest_filename)

                print(f"  OOF SHAP data saved (run={run_number}):")
                print(f"   - SHAP values: {shap_filename} (shape: {final_oof_shap_values.shape})")
                print(f"   - X_test: {xtest_filename} (shape: {final_oof_xtest.shape})")
            # --- End of OOF saving ---


            # --- Collect results ---
            # Errors when collecting results mean the final data frames would be incomplete,
            # so the exception is propagated.
            # 1. Scores
            # (If the fold loop finished without errors, every metric has 5 values = n_splits.)
            for metric, values in fold_scores.items():
                scores_list.append({
                    'ballot': ballot,
                    'model': model_name,
                    'year': year,
                    'run': run_number,
                    'metric': metric,
                    'avg_score': np.mean(values),
                    'std_score': np.std(values)
                })

            # 2. Feature importance (SHAP)
            # (If the fold loop finished without errors, fold_importances has 5 entries = n_splits.)
            importances_array = np.array(fold_importances)
            avg_importances = np.nanmean(importances_array, axis=0)

            # A feature whose SHAP importance is NaN in every fold indicates a SHAP problem;
            # rather than silently dropping the feature from the results,
            # raise an error.
            for feature, importance in zip(feature_names, avg_importances):
                if np.isnan(importance):
                    raise ValueError(
                        f"{dataset_identifier} - SHAP importance of feature '{feature}' "
                        f"is NaN in every fold."
                    )
                importances_list.append({
                    'ballot': ballot,
                    'model': model_name,
                    'year': year,
                    'run': run_number,
                    'feature': feature,
                    'importance': importance
                })

            # 3. Dimensions and features used
            dimensions_list.append({
                'ballot': ballot,
                'model': model_name,
                'year': year,
                'run': run_number,
                'n_rows': len(X),
                'n_cols': len(feature_names)
            })

            feature_usage_list.append({
                'ballot': ballot,
                'model': model_name,
                'year': year,
                'run': run_number,
                'features_used': ','.join(feature_names),
                'con_features_used': ','.join([f for f in feature_names if f not in categorical_cols]),
                'nom_features_used': ','.join(categorical_cols)
            })
            # --- End of collecting results ---

        # --- End of year loop ---
    # --- End of (ballot, model) loop ---

    return pd.DataFrame(scores_list), pd.DataFrame(importances_list), pd.DataFrame(dimensions_list), pd.DataFrame(feature_usage_list)


# --- Repeat run_single_ml_iteration_weighted_top n_runs times and combine the results ---
def run_ml_multiple_times_weighted(
    data_for_analysis_dict, # input data: {(ballot, model): {year: DataFrame}}
    target_col='partyid_1',
    n_runs=3,
    use_weights=True,
    output_folder_name=None # OOF SHAP folder (passed on to run_single_ml_iteration_weighted_top)
):
    """
    Run run_single_ml_iteration_weighted_top n_runs times with random_seed = 0..n_runs-1
    and return the mean/SD of all runs by (ballot, year, model, metric).
    """
    all_scores_list = []
    all_importances_list = []
    all_dimensions_list = []
    all_feature_usage_list = []

    for i in range(n_runs):
        print(f"\n{'='*30}\n         RUN {i+1} / {n_runs}         \n{'='*30}")

        run_scores_df, run_importances_df, run_dimensions_df, run_feature_usage_df = run_single_ml_iteration_weighted_top(
            data_for_analysis_dict=data_for_analysis_dict,
            target_col=target_col,
            random_seed=i, # different seed per run, so CV splits and model initialization differ
            run_number=i+1,
            use_weights=use_weights,
            output_folder_name=output_folder_name
        )

        all_scores_list.append(run_scores_df)
        all_importances_list.append(run_importances_df)
        all_dimensions_list.append(run_dimensions_df)
        all_feature_usage_list.append(run_feature_usage_df)
        print(f"--- Run {i+1} completed. ---")

    # Combine the runs (skipping empty results)
    scores_to_concat = [df for df in all_scores_list if isinstance(df, pd.DataFrame) and not df.empty]
    importances_to_concat = [df for df in all_importances_list if isinstance(df, pd.DataFrame) and not df.empty]
    dims_to_concat = [df for df in all_dimensions_list if isinstance(df, pd.DataFrame) and not df.empty]
    usage_to_concat = [df for df in all_feature_usage_list if isinstance(df, pd.DataFrame) and not df.empty]

    if not scores_to_concat:
        print("!! Warning: No scores generated. Returning empty results.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    all_scores_df = pd.concat(scores_to_concat, ignore_index=True)
    all_importances_df = pd.concat(importances_to_concat, ignore_index=True) if importances_to_concat else pd.DataFrame()
    all_dimensions_df = pd.concat(dims_to_concat, ignore_index=True) if dims_to_concat else pd.DataFrame()
    all_feature_usage_df = pd.concat(usage_to_concat, ignore_index=True) if usage_to_concat else pd.DataFrame()

    # Final summary: mean/SD over runs by (ballot, year, model, metric).
    final_summary = pd.DataFrame()
    if not all_scores_df.empty:
        # A failing aggregation (groupby/agg) means all_scores_df is malformed (e.g. missing columns),
        # so the exception is propagated.
        final_summary = all_scores_df.groupby(['ballot', 'year', 'model', 'metric'])['avg_score'].agg(['mean', 'std']).reset_index()
        final_summary = final_summary.rename(columns={'mean': 'final_avg_score', 'std': 'final_std_score'})
        print(f"\n\n{'='*40}\n     FINAL AVERAGED RESULTS ({n_runs} RUNS)     \n{'='*40}")
        auc_summary = final_summary[final_summary['metric'] == 'roc_auc_ovr']
        if not auc_summary.empty:
            print(auc_summary.round(4).to_markdown(index=False))
        else:
            print("No ROC AUC scores.")

    return all_scores_df, all_importances_df, final_summary, all_dimensions_df, all_feature_usage_df


# --- Summary and saving ---
def compare_model_performance(scores_df, metric='roc_auc_ovr'):
    """Mean/SD of `metric` by (ballot, year, model), sorted."""
    print(f"\n{'='*30}\n   Model performance ({metric})   \n{'='*30}")

    if scores_df.empty or metric not in scores_df['metric'].values:
        print("!! Warning: Scores DataFrame empty/metric not found.")
        return pd.DataFrame(columns=['ballot', 'year', 'model', 'avg_score_mean', 'avg_score_std'])

    metric_df = scores_df[scores_df['metric'] == metric].copy()
    performance_summary = metric_df.groupby(['ballot', 'year', 'model'])['avg_score'].agg(['mean', 'std']).reset_index()
    performance_summary = performance_summary.rename(columns={'mean': 'avg_score_mean', 'std': 'avg_score_std'})
    performance_summary = performance_summary.sort_values(by=['ballot', 'year', 'avg_score_mean'], ascending=[True, True, False])
    print(performance_summary.round(4))
    return performance_summary

def analyze_feature_frequency(importance_df, top_n=5):
    """
    For each (ballot, model, run, year), take the top_n features by importance and count
    how often each feature is in the top N (frequency_top_n) and how often it is used at all (total_occurrences);
    frequency_normalized = frequency_top_n / total_occurrences.
    A high ratio means that whenever the feature is used in that (ballot, model) group,
    it almost always ranks near the top.
    """

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty/missing importance column. Skipping frequency analysis.")
        return pd.DataFrame(), pd.DataFrame()

    # --- 0. Denominator base ---
    # With the {(ballot, model): {year: df}} structure, 'ballot' and 'year' columns always exist.
    denominator_base_df = importance_df.copy()
    top_n_group_keys = ['run', 'ballot', 'year', 'model']

    # --- 1. Top-N features (numerator base) ---
    # (rows used for 'frequency_top_n')
    top_features = importance_df.sort_values('importance', ascending=False).groupby(top_n_group_keys).head(top_n)

    numerator_base_df = top_features

    # --- 3. Overall frequency (not split by ballot/model) ---
    # Counts only; no normalization because groups are not split.
    overall_analysis = top_features.groupby('feature')['importance'].agg(
        frequency=('count'),
        importance_mean=('mean')
    ).reset_index().sort_values(by='frequency', ascending=False)

    # --- 4. By group: numerator ---
    grouped_analysis_numerator = numerator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        frequency_top_n=('count'), # (A) times in the top N
        importance_mean=('mean')
    ).reset_index()

    # --- 5. By group: denominator ---
    grouped_analysis_denominator = denominator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        total_occurrences=('count') # (B) times the variable was used
    ).reset_index()

    # --- 6. Merge (A) and (B), normalized frequency ---
    grouped_analysis = pd.merge(
        grouped_analysis_numerator,
        grouped_analysis_denominator,
        on=['ballot', 'model', 'feature'],
        how='left' # keyed on the variables in the top N
    )

    # (A) / (B)
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_top_n'].astype(float) / grouped_analysis['total_occurrences']
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_normalized'].fillna(0) # guard against NaN

    # --- 7. Sort by normalized frequency ---
    grouped_analysis = grouped_analysis.sort_values(
        by=['ballot', 'model', 'frequency_normalized'],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    # --- 8. Print results by (ballot, model) ---
    print("\n--- Grouped Frequency Analysis (Normalized) ---")
    for name, group in grouped_analysis.groupby(['ballot', 'model']):
        ballot_name, model_name = name
        print(f"\n[Group: {ballot_name} - {model_name}]")
        # Columns to show
        print(group[[
            'feature',
            'frequency_normalized',
            'frequency_top_n',
            'total_occurrences',
            'importance_mean'
        ]].head(5).to_markdown(index=False, floatfmt=".4f"))

    return overall_analysis, grouped_analysis

def analyze_temporal_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by year (score + rank)   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping temporal analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for temporal analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_model = {}
    for model_name in avg_importance_df['model'].unique():
        model_df = avg_importance_df[avg_importance_df['model'] == model_name]
        for ballot_type in sorted(model_df['ballot'].unique()):
            ballot_df = model_df[model_df['ballot'] == ballot_type]
            # A failing pivot (e.g. duplicated (feature, year) pairs) means importance_df
            # is malformed, so the exception is allowed to stop the analysis.
            pivot_table = ballot_df.pivot_table(index='feature', columns='year', values=['importance', 'rank'])
            pivot_table = pivot_table.swaplevel(0, 1, axis=1).sort_index(axis=1)
            results_by_model[f"Temporal_{model_name}_{ballot_type}"] = pivot_table
    return results_by_model

def analyze_cross_model_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by model and year   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping cross-model analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for cross-model analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_ballot = {}
    for ballot_type in sorted(avg_importance_df['ballot'].unique()):
        ballot_df = avg_importance_df[avg_importance_df['ballot'] == ballot_type]
        # A failing pivot (e.g. duplicated (feature, year, model) triples) means importance_df
        # is malformed, so the exception is allowed to stop the analysis.
        pivot_table = ballot_df.pivot_table(index='feature', columns=['year', 'model'], values=['importance', 'rank'])
        pivot_table.columns = [f"{year}_{model}_{val}" for val, year, model in pivot_table.columns]
        pivot_table = pivot_table.sort_index(axis=1)
        results_by_ballot[f"CrossModel_{ballot_type}"] = pivot_table
    return results_by_ballot

def save_results_to_excel(results_dict, filename="ml_analysis_summary.xlsx"):
    """Save each item of results_dict as one sheet of an Excel file."""
    with pd.ExcelWriter(filename) as writer:
        for sheet_name, df in results_dict.items():
             if isinstance(df, pd.DataFrame):
                # Ensure sheet names are valid
                safe_sheet_name = str(sheet_name).replace('[','').replace(']','').replace('*','').replace(':','').replace('?','').replace('/','\\')
                safe_sheet_name = safe_sheet_name[:31] # Max length limit

                if isinstance(df.index, (pd.MultiIndex, pd.Index)) and df.index.name is not None:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=True)
                else:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
    print(f"\n  All results saved to '{filename}'.")


# --- Full workflow: training -> performance/importance summaries -> Excel/Parquet ---
def run_analysis_for_target_weighted_selective(
    df_dict,          # input data: {(ballot, model): {year: DataFrame}}
    df_name,          # name used for the output files and the OOF SHAP folder (e.g. "M3_Top40")
    target_col,       # target column
    n_runs            # number of repetitions passed to run_ml_multiple_times_weighted
):
    """
    Run the full workflow for target_col (training -> performance comparison -> feature importance
    -> changes by year/model) and save the results to one Excel file and three parquet files.
    df_name is used for both the OOF SHAP folder and the output file names.
    """
    print(f"\n{'='*60}")
    print(f"  Starting Workflow for Target Variable: '{target_col}' ({n_runs} runs)")
    print(f"  Data Source (df_name): {df_name}")
    print(f"{'='*60}")

    # Train/evaluate n_runs times; the OOF SHAP values of run 1 are saved in the df_name folder.
    all_scores, all_importances, final_summary, all_dimension, all_feature_usage = run_ml_multiple_times_weighted(
        data_for_analysis_dict=df_dict,
        target_col=target_col,
        n_runs=n_runs,
        use_weights=True, # GSS weights
        output_folder_name=os.path.join(OUT_DIR, 'shap', df_name)
    )

    performance_comparison_df = compare_model_performance(all_scores, metric='roc_auc_ovr')

    # all_importances has 'ballot', 'year', 'model' and 'run' columns,
    # as analyze_feature_frequency expects.
    overall_freq_df, grouped_freq_df = analyze_feature_frequency(all_importances, top_n=20)
    temporal_results = analyze_temporal_change(all_importances)
    cross_model_results = analyze_cross_model_change(all_importances)

    print(f"\n--- [{target_col} - {df_name}] Collecting results for the Excel file... ---\n")

    # Mean sample size (n_rows) and number of features (n_cols) over runs by (ballot, year, model).
    unique_dimensions_df = pd.DataFrame()
    if not all_dimension.empty:
        # A failing aggregation (groupby/mean) means all_dimension is malformed,
        # so the exception is propagated.
        avg_dims = all_dimension.groupby(['ballot', 'year', 'model'])[['n_rows', 'n_cols']].mean().reset_index()
        unique_dimensions_df = avg_dims

    # Log of the features actually used; identical across runs, so run 1 is used.
    unique_feature_usage_df = pd.DataFrame()
    if not all_feature_usage.empty:
        unique_feature_usage_df = all_feature_usage[all_feature_usage['run'] == 1].drop(columns=['run']).reset_index(drop=True)


    # Sheets of the Excel file: summary/dimensions/performance/feature frequency + changes by year/model
    excel_sheets = {
        "0_Final_Summary": final_summary,
        "1_Dataset_Dimensions": unique_dimensions_df,
        "2_Performance_Compare": performance_comparison_df,
        "3_Feature_Freq_Overall": overall_freq_df,
        "4_Feature_Freq_Grouped": grouped_freq_df,
        "5_Feature_Usage_Log": unique_feature_usage_df,
        **temporal_results,
        **cross_model_results
    }

    # df_name (e.g. "M3_Top40") in the file names identifies the feature set.
    os.makedirs(OUT_DIR, exist_ok=True)
    excel_filename = os.path.join(OUT_DIR, f"7_FINAL_{df_name}_{target_col}_analysis_results_weighted.xlsx")
    importance_filename = os.path.join(OUT_DIR, f"7_FINAL_{df_name}_{target_col}_analysis_all_importance_weighted.parquet")
    scores_filename = os.path.join(OUT_DIR, f"7_FINAL_{df_name}_{target_col}_analysis_all_scores_weighted.parquet")
    usage_log_filename = os.path.join(OUT_DIR, f"7_FINAL_{df_name}_{target_col}_analysis_all_feature_usage.parquet")

    save_results_to_excel(excel_sheets, filename=excel_filename)

    # all_importances/all_scores/all_feature_usage have one row per (ballot x model x year x run),
    # so they are saved as parquet files rather than Excel.
    print(f"\n--- [{target_col} - {df_name}] Saving raw results to parquet... ---")
    if not all_importances.empty: all_importances.to_parquet(importance_filename)
    if not all_scores.empty: all_scores.to_parquet(scores_filename)
    if not all_feature_usage.empty: all_feature_usage.to_parquet(usage_log_filename)

    print(f"  [{target_col} - {df_name}] All files saved.")
    print(f"{'-'*60}\n")

## Run

In [ ]:
# ===== Settings =====
DF_NAME      = 'M3_Top40'   # used in the output file names and the OOF folder (M3 top-40 selection)
N_ITERATIONS = 20          # number of repetitions (can be reduced for a test run)

if 'run_analysis_for_target_weighted_selective' not in globals():
    raise RuntimeError('Run the prediction model cell first.')

run_analysis_for_target_weighted_selective(
    df_dict=data_for_analysis_race,
    df_name=DF_NAME,
    target_col='race_binary',
    n_runs=N_ITERATIONS,
)

run_analysis_for_target_weighted_selective(
    df_dict=data_for_analysis_sex,
    df_name=DF_NAME,
    target_col='sex',
    n_runs=N_ITERATIONS,
)

print('\nFinal race_binary / sex analyses finished.')